In [0]:
dbutils.widgets.text("p_batch_id","")
v_batch_id=dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/03.SilverHelper

In [0]:

bronze_table=f"{catalog_name}.{bronze_schema}.races"
silver_table=f"{catalog_name}.{silver_schema}.races"


In [0]:

silver_table

In [0]:
%sql
describe history formula1_catalog.bronze.races

In [0]:
# spark.read for aditonal options to read table data
#ciucuits_df=spark.read.option('versionAsOf',0).table(bronze_table)

In [0]:
races_df=spark.table(bronze_table)

In [0]:
import pyspark.sql.functions as F
races_df=spark.table(bronze_table).filter((F.col("batch_id")== v_batch_id))


In [0]:
from pyspark.sql import functions as F
races_df_selected=races_df.select(
    F.col("season"),
    F.col("round"),
    F.col("raceName"),
    F.col("date"),
    F.col("circuitId"),
    F.col("ingestion_timestamp"),
    F.col("source_file"),
    F.col("batch_id")
    )


In [0]:
races_renamed_df=(
    races_df_selected
        .withColumnsRenamed ({
                     "circuitId":"circuit_id",
                     "raceName":"race_name",
                     "date":"race_date"
                     })  
                    
)

In [0]:
#circuits_renamed_nulldroped_df=circuits_renamed_df.filter("circuit_id is not  null")
#circuits_renamed_nulldroped_df=circuits_renamed_df.filter(circuits_renamed_df['circuit_id'].isNotNull())
races_renamed_nulldroped_df=races_renamed_df.filter(
    F.col('circuit_id').isNotNull()
)

In [0]:
display(races_renamed_nulldroped_df.count())
display(races_renamed_df.count())


In [0]:
#circuits_distinct_df=circuits_renamed_nulldroped_df.distinct()
races_distinct_df=races_renamed_df.dropDuplicates(["season", "round"])
display(races_distinct_df)

In [0]:

duplicates = races_renamed_df[["season", "round"]].groupBy("season", "round").count().filter("count > 1")
display(duplicates)

In [0]:
from pyspark.sql.functions import initcap
races_final_df=(races_distinct_df
    .withColumn('race_name',F.initcap(F.col('race_name')))
 )

In [0]:
display(races_final_df)

In [0]:
from pyspark.sql.functions import monotonically_increasing_id

df_with_id = races_final_df.withColumn("row_id", monotonically_increasing_id())


In [0]:
display(df_with_id)

In [0]:
# (
#     races_final_df
#         .write
#         .format("delta")
#         .mode('overwrite')
#         .saveAsTable(silver_table)
# )

In [0]:
write_to_silver(
    input_df=races_final_df,
    target_table=silver_table,
    merge_condition="t.season=s.season AND t.round=s.round",
    columns_to_update=[
        "season",
        "round",
        "race_name",
        "race_date",
        "ingestion_timestamp",
        "batch_id"
    ]
)

In [0]:
%sql
select * from formula1_incr_catalog.silver.races
--drop table formula1_incr_catalog.silver.races